# CNNs Revisited: ConvNeXt On Food-101 Walkthrough

This notebook is the interactive route for the CNNs revisited chapter. `convnext_food101_pytorch.py` remains the canonical implementation for repeatable runs, metadata, metrics, checkpoints, and artifacts.

The default cells start in synthetic smoke mode so readers can inspect the data path, classifier head, freezing policy, and saved artifacts without downloading Food-101 or pretrained weights. Move to the full Food-101 command only after the smoke path and artifact contract are clear.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Point At The Companion Code

The setup cell finds the chapter directory, imports the canonical script as a module, and defines common paths. This lets the notebook inspect the same functions used by command-line runs instead of maintaining a second implementation.

In [ ]:
from pathlib import Path
import csv
import json
import shlex
import subprocess
import sys

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = None
    Markdown = None
    display = None


def find_code_dir():
    script_name = 'convnext_food101_pytorch.py'
    chapter_name = 'chapter_cnn_revisited'
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / 'code' / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(
        'Run this notebook from the repository root or chapter_cnn_revisited.'
    )


CODE_DIR = find_code_dir()
REPO_ROOT = CODE_DIR.parent
SCRIPT = CODE_DIR / 'convnext_food101_pytorch.py'
RUNS_DIR = CODE_DIR / 'runs'

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import convnext_food101_pytorch as convnext_script

print('Notebook directory:', CODE_DIR)
print('Canonical script:', SCRIPT)

## 2. Run A Dependency Check

The dependency check reports whether PyTorch, TorchVision, and related runtime pieces are available in the selected kernel. It uses `--allow-missing-deps` so readers on a lightweight environment get actionable install guidance instead of a notebook crash.

In [ ]:
check_cmd = [
    sys.executable,
    str(SCRIPT),
    '--check-deps',
    '--allow-missing-deps',
]
print(' '.join(shlex.quote(part) for part in check_cmd))
result = subprocess.run(check_cmd, cwd=CODE_DIR, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'dependency check failed with exit code {result.returncode}')

## 3. Configure A Small Interactive Run

The defaults use synthetic data, small images, tiny train and validation limits, and no in-notebook training. Keep `SMOKE_MODE = True` while learning the workflow; switch to real Food-101 paths only when the dependency check, data shape, and artifact steps are understood.

In [ ]:
SMOKE_MODE = True
MODEL_VARIANT = 'convnext_tiny'
IMAGE_SIZE = 64 if SMOKE_MODE else 224
RESIZE_SIZE = 72 if SMOKE_MODE else 256
BATCH_SIZE = 4 if SMOKE_MODE else 32
SEED = 42
NUM_CLASSES = 101
DEVICE = None
DATA_ROOT = None
LIMIT_TRAIN = 8 if SMOKE_MODE else None
LIMIT_VAL = 4 if SMOKE_MODE else None

RUN_INTERACTIVE_TRAINING = False

config = {
    'smoke_mode': SMOKE_MODE,
    'model_variant': MODEL_VARIANT,
    'image_size': IMAGE_SIZE,
    'resize_size': RESIZE_SIZE,
    'batch_size': BATCH_SIZE,
    'seed': SEED,
    'num_classes': NUM_CLASSES,
    'data_root': DATA_ROOT,
    'limit_train': LIMIT_TRAIN,
    'limit_val': LIMIT_VAL,
}
print(json.dumps(config, indent=2))

## 4. Import PyTorch And Inspect The Device

This cell imports the optional torch stack through the companion script and prints the resolved device. Record the device and package versions with reportable runs, because pretrained vision experiments can differ across CPU, CUDA, and mixed-precision environments.

In [ ]:
try:
    torch, torchvision, transforms = convnext_script.import_torch_stack()
except RuntimeError as exc:
    torch = None
    torchvision = None
    transforms = None
    print('PyTorch or TorchVision is not available in this kernel.')
    print('Install the transfer-learning companion-code stack or switch to that environment.')
    print(exc)
else:
    convnext_script.print_dependency_summary(torch, torchvision)
    device = convnext_script.resolve_device(torch, DEVICE)
    print('selected_device=', device)

## 5. Build Synthetic Data Loaders

The synthetic loaders exercise the same batch plumbing as the real Food-101 dataset: image tensors, integer labels, class names, and path-like identifiers. Inspect the printed shape table before changing image size, batch size, or dataset location.

In [ ]:
def markdown_table(headers, rows):
    header = '| ' + ' | '.join(headers) + ' |'
    divider = '| ' + ' | '.join('---' for _ in headers) + ' |'
    body = ['| ' + ' | '.join(str(value) for value in row) + ' |' for row in rows]
    return '\n'.join([header, divider, *body])


def show_markdown(text):
    if Markdown is not None and display is not None:
        display(Markdown(text))
    else:
        print(text)


if torch is None:
    print('Install PyTorch to inspect synthetic batches.')
else:
    torch.manual_seed(SEED)
    if SMOKE_MODE:
        train_dataset = convnext_script.SyntheticFood101Dataset(
            torch, size=LIMIT_TRAIN or 8, image_size=IMAGE_SIZE, num_classes=NUM_CLASSES, seed=SEED
        )
        val_dataset = convnext_script.SyntheticFood101Dataset(
            torch, size=LIMIT_VAL or 4, image_size=IMAGE_SIZE, num_classes=NUM_CLASSES, seed=SEED + 1
        )
        class_names = [f'class_{index:03d}' for index in range(NUM_CLASSES)]
        data_root = 'synthetic smoke data'
    else:
        root_arg = Path(DATA_ROOT).expanduser() if DATA_ROOT else None
        data_root = convnext_script.resolve_data_root(root_arg)
        train_transform, eval_transform = convnext_script.make_transforms(
            transforms, image_size=IMAGE_SIZE, resize_size=RESIZE_SIZE
        )
        train_records, val_records, _test_records, class_names = convnext_script.load_food101_records(
            data_root,
            seed=SEED,
            valid_pct_value=0.2,
            limit_train=LIMIT_TRAIN,
            limit_val=LIMIT_VAL,
            limit_test=None,
        )
        train_dataset = convnext_script.Food101ImageDataset(train_records, train_transform)
        val_dataset = convnext_script.Food101ImageDataset(val_records, eval_transform)
    train_loader = convnext_script.make_loader(
        torch, train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )
    val_loader = convnext_script.make_loader(
        torch, val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )
    images, labels, paths = convnext_script.unpack_batch(next(iter(train_loader)))
    rows = [
        ['data root', data_root],
        ['class count', len(class_names)],
        ['train examples', len(train_dataset)],
        ['validation examples', len(val_dataset)],
        ['batch image shape', tuple(images.shape)],
        ['batch labels', labels.tolist()],
        ['first paths', ', '.join(list(paths)[:2])],
    ]
    show_markdown(markdown_table(['Field', 'Value'], rows))

### Reader Checkpoint

After the previous cell runs, confirm that the printed paths, shapes, commands, or tables match the section description before moving on. If this checkpoint fails in a public-repo environment, fix dependencies, data paths, or guarded flags before starting longer runs.

## 6. Inspect The ConvNeXt Head And Freezing Policy

This cell replaces the classifier for the current class count and compares trainable parameter counts for the frozen-head and fine-tuning stages. The point is to connect the chapter discussion of transfer learning to the actual parameters that will receive gradients.

In [ ]:
def parameter_counts(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable


if torch is None:
    print('Install PyTorch to build and inspect ConvNeXt.')
else:
    model_num_classes = len(class_names) if 'class_names' in globals() else NUM_CLASSES
    model, weights_name = convnext_script.build_model(
        torch,
        torchvision,
        variant=MODEL_VARIANT,
        pretrained=False,
        num_classes=model_num_classes,
    )
    total_parameters, initial_trainable = parameter_counts(model)
    classifier = model.classifier[-1]

    convnext_script.set_trainable(model, train_body=False)
    _total, frozen_trainable = parameter_counts(model)
    convnext_script.set_trainable(model, train_body=True)
    _total, unfrozen_trainable = parameter_counts(model)

    rows = [
        ['weights', weights_name],
        ['classifier input features', classifier.in_features],
        ['classifier output classes', classifier.out_features],
        ['total parameters', f'{total_parameters:,}'],
        ['trainable after build', f'{initial_trainable:,}'],
        ['trainable during frozen-head stage', f'{frozen_trainable:,}'],
        ['trainable during fine-tuning stage', f'{unfrozen_trainable:,}'],
    ]
    show_markdown(markdown_table(['Field', 'Value'], rows))

## 7. Optional In-Notebook Training

In-notebook training is disabled by default because the script is the reproducible route for metrics and artifacts. Enable this only for a local sanity check after the synthetic loaders and model head look correct, and treat any result here as a workflow check rather than a reportable Food-101 score.

In [ ]:
if torch is None:
    print('Install PyTorch to train inside the notebook.')
elif RUN_INTERACTIVE_TRAINING:
    output_dir = RUNS_DIR / 'notebook-interactive-run'
    output_dir.mkdir(parents=True, exist_ok=True)
    device = convnext_script.resolve_device(torch, DEVICE)
    model.to(device)
    best_state = {}
    all_metrics = []

    convnext_script.set_trainable(model, train_body=False)
    all_metrics.extend(
        convnext_script.train_stage(
            torch,
            model,
            stage='frozen_head',
            epochs=1,
            lr=3e-4,
            weight_decay=0.05,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            mixed_precision=False,
            checkpoint_dir=output_dir,
            best_state=best_state,
        )
    )
    convnext_script.write_history(output_dir / 'history.csv', all_metrics)
    print('best_state:', best_state)
else:
    print('Set RUN_INTERACTIVE_TRAINING = True to train this in-notebook model.')

## 8. Run The Canonical Quick Command

The quick command shells out to `convnext_food101_pytorch.py` with synthetic data and plot saving enabled. This is the public-repo smoke path: it confirms that the script can write metadata, metrics, history, checkpoints, and plots without requiring the full dataset.

In [ ]:
smoke_output_dir = RUNS_DIR / 'notebook-smoke-run'
smoke_cmd = [
    sys.executable,
    str(SCRIPT),
    '--quick',
    '--save-plots',
    '--output-dir',
    str(smoke_output_dir),
    '--allow-missing-deps',
]
print(' '.join(shlex.quote(part) for part in smoke_cmd))
result = subprocess.run(smoke_cmd, cwd=CODE_DIR, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'quick command failed with exit code {result.returncode}')

## 9. Read The Run Artifacts

After the quick command, inspect the written JSON, CSV, and optional PNG artifacts. The synthetic accuracy values are not model-quality evidence; the important check is that the files have the expected fields and can support a later reportable run.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def read_csv_rows(path):
    with Path(path).open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


metrics_path = smoke_output_dir / 'metrics.json'
metadata_path = smoke_output_dir / 'metadata.json'
if not metrics_path.exists() or not metadata_path.exists():
    print('No smoke artifacts found. If dependencies were missing, the quick command exited after printing install help.')
else:
    metadata = read_json(metadata_path)
    metrics = read_json(metrics_path)
    summary = {
        'device': metadata['device'],
        'model_variant': metadata['model_variant'],
        'train_examples': metadata['train_examples'],
        'validation_examples': metadata['validation_examples'],
        'best_validation_accuracy': metrics.get('best_validation_accuracy'),
        'validation_accuracy': metrics.get('validation_accuracy'),
        'test_accuracy': metrics.get('test_accuracy'),
    }
    print(json.dumps(summary, indent=2))

    history_path = smoke_output_dir / 'history.csv'
    if history_path.exists():
        print('history:')
        for row in read_csv_rows(history_path):
            print(row)

    history_plot = smoke_output_dir / 'history_plot.png'
    if history_plot.exists() and Image is not None and display is not None:
        display(Image(filename=str(history_plot)))

## 10. Preview The Committed Reference Evidence

The compact reference artifacts mirror the evidence cited by the chapter without bundling the full training run. Use this cell to inspect validation and test metrics, weak classes, and the training curve before deciding whether a new full run is necessary.

In [ ]:
reference_dir = CODE_DIR / 'reference_artifacts' / 'food101-convnext-reference'
reference_metrics_path = reference_dir / 'metrics.json'
if not reference_metrics_path.exists():
    print('No committed reference metrics found at', reference_metrics_path)
else:
    reference_metrics = read_json(reference_metrics_path)
    selected = {
        'best_validation_accuracy': reference_metrics.get('best_validation_accuracy'),
        'validation_accuracy': reference_metrics.get('validation_accuracy'),
        'test_accuracy': reference_metrics.get('test_accuracy'),
        'test_top5_accuracy': reference_metrics.get('test_top5_accuracy'),
        'elapsed_seconds': reference_metrics.get('elapsed_seconds'),
    }
    print(json.dumps(selected, indent=2))

    per_class_path = reference_dir / 'test_per_class_accuracy.csv'
    if per_class_path.exists():
        rows = read_csv_rows(per_class_path)
        rows = sorted(rows, key=lambda row: float(row['accuracy']))
        show_markdown(
            'Lowest test per-class accuracies:\n'
            + markdown_table(
                ['Class', 'Correct', 'Total', 'Accuracy'],
                [
                    [row['class_name'], row['correct'], row['total'], row['accuracy']]
                    for row in rows[:8]
                ],
            )
        )

    reference_plot = reference_dir / 'history_plot.png'
    if reference_plot.exists() and Image is not None and display is not None:
        display(Image(filename=str(reference_plot)))

## 11. Full Food-101 Baseline Command

This cell prints the full command shape for a real Food-101 baseline and leaves execution disabled. Review the dataset path, image size, batch size, learning rates, mixed-precision setting, output directory, and compute budget before setting `RUN_FULL_FOOD101 = True`.

In [ ]:
RUN_FULL_FOOD101 = False
full_output_dir = RUNS_DIR / 'convnext-tiny-food101'
full_cmd = [
    sys.executable,
    str(SCRIPT),
    '--data-root',
    '~/.fastai/data/food-101',
    '--model-variant',
    'convnext_tiny',
    '--image-size',
    '224',
    '--resize-size',
    '256',
    '--batch-size',
    '32',
    '--freeze-epochs',
    '1',
    '--epochs',
    '4',
    '--head-lr',
    '3e-4',
    '--fine-tune-lr',
    '1e-5',
    '--weight-decay',
    '0.05',
    '--mixed-precision',
    '--save-plots',
    '--output-dir',
    str(full_output_dir),
]
print(' '.join(shlex.quote(part) for part in full_cmd))

if RUN_FULL_FOOD101:
    subprocess.run(full_cmd, cwd=CODE_DIR, check=True)
else:
    print('Review the command first. Set RUN_FULL_FOOD101 = True when you intend a full run.')

## 12. Reserved Test Evaluation

Keep reserved test evaluation off while tuning. Run this cell only once for the final selected configuration, after validation evidence has fixed the model and training recipe, so the test set remains an honest final estimate.

In [ ]:
EVALUATE_RESERVED_TEST = False
final_test_cmd = full_cmd + ['--evaluate-test']
print(' '.join(shlex.quote(part) for part in final_test_cmd))

if EVALUATE_RESERVED_TEST:
    subprocess.run(final_test_cmd, cwd=CODE_DIR, check=True)
else:
    print('Leave EVALUATE_RESERVED_TEST = False until the final selected configuration.')

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.